In [ ]:
import pandas as pd

train_df = pd.read_csv("")
test_df = pd.read_csv("")

In [3]:

train_df["side"] = train_df["side"].map({
    "Blue": 1,
    "Red": 0
})

test_df["side"] = test_df["side"].map({
    "Blue": 1,
    "Red": 0
})

prediction_df = test_df.copy()

drop_columns = [
    "Unnamed: 0",
    "date",
    "league",
    "teamname",
    'avg_deaths_before',
    'avg_heralds_before',
    'avg_firsttower_before',
    'avg_assists_before',
    'avg_firstblood_before'
]

train_df = train_df.drop(columns=drop_columns)
test_df = test_df.drop(columns=drop_columns)

X_train = train_df.drop(columns=["result"])
y_train = train_df["result"]

X_test = test_df.drop(columns=["result"])
y_test = test_df["result"]

print(X_train.shape)
print(X_test.shape)




(236, 13)
(160, 13)


In [4]:
# Random Forest

from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=3,
    min_samples_leaf=10,
    random_state=42
)

model.fit(X_train, y_train)


# Evaluation
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")
print(confusion_matrix(y_test, y_pred))

importance = (
    pd.DataFrame({
        "Feature": X_train.columns,
        "Importance": model.feature_importances_
    })
    .sort_values("Importance", ascending=False)
)

print(importance)

# Prediction
latest_stats = (
    prediction_df
    .sort_values("date")
    .groupby("teamname")
    .tail(1)
)

X_latest = latest_stats[X_train.columns]
latest_stats["Win Probability"] = model.predict_proba(X_latest)[:, 1]

latest_stats = latest_stats.sort_values(
    "Win Probability",
    ascending=False
)

print(latest_stats[
    ["teamname", "Win Probability"]
])


Accuracy: 0.544
[[42 19]
 [54 45]]
                    Feature  Importance
12    avg_csdiffat15_before    0.113632
11    avg_xpdiffat15_before    0.109894
6    avg_visionscore_before    0.107231
10  avg_golddiffat15_before    0.096933
2          avg_kills_before    0.087592
9           avg_cspm_before    0.084883
8     avg_earned gpm_before    0.078749
7     avg_earnedgold_before    0.072139
1           win_rate_before    0.071921
5         avg_towers_before    0.067852
3        avg_dragons_before    0.066121
4         avg_barons_before    0.038828
0                      side    0.004225
                teamname  Win Probability
121                   T1         0.615026
11       Bilibili Gaming         0.605721
63   Hanwha Life Esports         0.557849
105                 LYON         0.491637
158   Team Secret Whales         0.476993
85          Karmine Corp         0.442542
145          Team Liquid         0.433294
28     Deep Cross Gaming         0.422596
53            G2 Esports   

In [5]:
# Log Regression

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    random_state=42,
    max_iter=1000
)

lr.fit(X_train_scaled, y_train)

y_pred = lr.predict(X_test_scaled)

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
importance = (
    pd.DataFrame({
        "Feature": X_train.columns,
        "Coefficient": lr.coef_[0]
    })
    .sort_values("Coefficient", key=abs, ascending=False)
)
print(importance)

X_latest = latest_stats[X_train.columns]
latest_stats["Win Probability"] = model.predict_proba(X_latest)[:, 1]

latest_stats = latest_stats.sort_values(
    "Win Probability",
    ascending=False
)

print(latest_stats[
    ["teamname", "Win Probability"]
])

Accuracy: 0.38125
[[56  5]
 [94  5]]
              precision    recall  f1-score   support

           0       0.37      0.92      0.53        61
           1       0.50      0.05      0.09        99

    accuracy                           0.38       160
   macro avg       0.44      0.48      0.31       160
weighted avg       0.45      0.38      0.26       160

                    Feature  Coefficient
7     avg_earnedgold_before    -0.844294
11    avg_xpdiffat15_before    -0.754921
12    avg_csdiffat15_before     0.725105
5         avg_towers_before     0.707867
9           avg_cspm_before    -0.597044
8     avg_earned gpm_before     0.593462
1           win_rate_before    -0.511137
3        avg_dragons_before     0.237851
4         avg_barons_before     0.146509
2          avg_kills_before     0.114191
10  avg_golddiffat15_before     0.114135
0                      side    -0.053055
6    avg_visionscore_before    -0.018192
                teamname  Win Probability
121                 

In [6]:
# Gradient Boosting

from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    random_state=42
)

gb.fit(X_train, y_train)

y_pred = gb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.36875
